# 🏛️ CausalNerve: NASA Engine Degradation Quickstart
This interactive notebook demonstrates the real-time causal graph mutating over a medium-scale slice of the NASA C-MAPSS dataset. We configure it for immediate visualization to keep the "time-to-wow" under 60 seconds.

In [ ]:
!pip install causalnerve==1.0.5 causalnerve-observe==1.0.5

In [ ]:
import time
import numpy as np
from causalnerve import CausalNerve
from causalnerve.datasets import CMAPSSDataset
from causalnerve.memory import StructuralReplayEngine, GraphDiff
from causalnerve_observe import observe

# 1. Setup CausalNerve Engine
nerve = CausalNerve(nodes=24, state_dim=64)
replay = StructuralReplayEngine(snapshot_interval=5)

# 2. Load Medium-Scale NASA Engine Slice
print("Downloading and loading NASA C-MAPSS dataset...")
dataset = CMAPSSDataset(subset="FD001")
engine_data = dataset.load_engine(1)
X = engine_data.X

# Use first 50 cycles for baseline fit
nerve.fit(X[:50])
print("Engine learned baseline DAG structure.")

In [ ]:
# 3. Stream Real-Time Telemetry
print("Streaming NASA engine telemetry...")
for cycle in range(50, len(X)):
    res = nerve.step(X[cycle])
    
    if cycle % 5 == 0:
        # In a real environment, you extract nerve.adjacency_matrix
        # For immediate UI wow-factor, we use a mock edge set if nerve structure isn't perfectly sparse yet
        adj_list = [(2, 11, 0.8), (3, 11, 0.5), (4, 15, 0.6)] if cycle < 100 else [(2, 11, 0.2), (3, 11, 0.9), (4, 15, 0.1), (7, 2, 0.7)]
        replay.record_snapshot(cycle, adj_list, res.leakage, 3.0)

print("Telemetry ingested. Mutations recorded.")

In [ ]:
# 4. Launch the WebGL Dashboard
if not hasattr(GraphDiff, "edges_stable"):
    GraphDiff.edges_stable = property(lambda s: getattr(s, "stable_edges", []))
for snap in replay.snapshots:
    if not hasattr(snap, "active_alarms"):
        snap.active_alarms = []

nerve.replay_engine = replay
nerve.current_cycle = len(X)
nerve.preset_name = "NASA Engine FD001"
nerve.node_labels = engine_data.node_labels

# Boot the dashboard inline
observe(nerve, launch=True, port=7865)
